# Gate 0: is the detector orientation-sensitive?

Drives `src/rotation.py` over one owner's half of the matrix. Every checkpoint that already
exists is scored against `rot90(x_test, k)` for k = 0, 1, 2, 3 - and the answer decides whether the D4 augmentation
ablation is worth 28 GPU-runs.

## Setup

Set `OWNER` in the next cell and nothing else. Same paths as `01_run_matrix.ipynb`: the cache
is read from the shared root, the checkpoints from **this account's own** Drive backup.

In [ ]:
# Re-run this after any runtime restart.
import os, sys, json, glob
from pathlib import Path

# A CPU runtime will run this - slowly, and `--device cuda` would fail outright. The check is
# cheap and the fallback message says what to do, because on a CPU runtime nvidia-smi is not
# installed at all and the bare "command not found" says nothing.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU on this runtime -> Runtime > Change runtime type > T4 GPU > Save, then re-run this cell (the restart unmounts Drive)"

from google.colab import drive
drive.mount('/content/drive')

# Who is running this notebook. Only this account's 28 checkpoints have weights here; the
# other 28 are skipped. That is normal here.
OWNER = "ido"

# Read root, shared between both accounts - identical bytes, nothing copied.
DRIVE = "/content/drive/MyDrive/university/deep_learning"
CACHE = f"{DRIVE}/cache"

# Write root, this account's OWN My Drive. Deliberately a different tree from CACHE so a
# view-only share cannot break it - see 01_run_matrix.ipynb, setup cell.
BACKUP = f"/content/drive/MyDrive/deep_learning_results/{OWNER}"

# Repo-relative: metrics.json comes from git and lists all 56 runs, so the loop sees the
# whole matrix and skips whatever has no local weights.
RESULTS = "results/runs"

assert os.path.isdir(DRIVE), (
    f"not found: {DRIVE}\nCheck it is mounted on THIS account: `ls /content/drive/MyDrive`."
)

print(f"owner   {OWNER}")
print("cache  <-", CACHE, "" if os.path.isdir(CACHE) else "  <- MISSING")
print("ckpts  <-", f"{BACKUP}/runs")

## Get the code

In [ ]:
# Clones on the first run, pulls on every later one.
![ -d /content/deep-learning ] || git clone -q https://github.com/Ido-Josephsberg/deep-learning.git /content/deep-learning
%cd /content/deep-learning
!git pull --ff-only
!git log --oneline -1

## ## Check whether this account's checkpoints actually there?


In [ ]:
found = sorted(glob.glob(f"{BACKUP}/runs/*/*/seed*/model.pt"))
metrics = sorted(glob.glob(f"{RESULTS}/*/*/seed*/metrics.json"))

by_strategy = {}
for path in found:
    by_strategy.setdefault(Path(path).parts[-4], []).append(path)

print(f"{len(metrics)} metrics.json from git (all 56 expected)")
print(f"{len(found)} checkpoints under {BACKUP}/runs (28 expected for one account)")
for strategy, paths in sorted(by_strategy.items()):
    print(f"  {strategy:<13}{len(paths):>3}")

if not found:
    print("\nNOTHING FOUND. Is OWNER right, and is Drive mounted on THIS account?")
elif len(found) < 28:
    print(f"\nOnly {len(found)} of 28 - the summary below will cover part of this half.")

## Run it


In [ ]:
gate0_cmd = (
    f'python -m src.rotation --cache-dir "{CACHE}" --ckpt-root "{BACKUP}/runs"'
    f' --owner {OWNER} --results-dir "{RESULTS}"'
)
!{gate0_cmd}

## Both halves together

Each account commits its own `rotation_sensitivity_<owner>.json`. Once both are on `main`,
this cell merges whatever is present and re-prints the tables and the verdict over all four
arms - the per-strategy table is the one that matters, because the mechanism is arm-specific:
`rescale` is where real images are aspect-distorted and generated ones are not, so an
orientation-signed real-class cue shows up there first.

In [ ]:
from src.rotation import format_report, verdict

rows = []
for path in sorted(glob.glob("results/figures/rotation_sensitivity_*.json")):
    part = json.loads(Path(path).read_text())
    rows += part
    print(f"{Path(path).name:<40}{len(part) // 4:>3} checkpoints")

arms = sorted({r["strategy"] for r in rows})
print(f"\n{len(rows) // 4} checkpoints over {len(arms)} arms: {', '.join(arms)}\n")
print(format_report(rows))
print("\nVERDICT:", verdict(rows))